# Training Sampler Diagnostics

This notebook reads the current `finetune.sh`, rebuilds the training sampler context, and visualizes what the model sees during training.

It supports:
- triplet mode and balanced mode
- raw vs augmented views
- anchor / positive / negative triplet panels
- balanced batch overviews
- sequence-aware metadata and pair-quality cache annotations when available

In [ ]:
from pathlib import Path
from pprint import pprint
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'contrastive_finetuning').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from contrastive_finetuning.training_sampler_diagnostics import (
    build_diagnostic_context,
    display_balanced_batch_overviews,
    display_triplet_examples,
    display_triplet_match_examples,
    summarize_context,
)

SHELL_PATH = REPO_ROOT / 'finetune.sh'
SEED = 0


In [ ]:
context = build_diagnostic_context(SHELL_PATH)
summary = summarize_context(context)
pprint(summary)


## Triplet Panels

Each panel shows:
- top row: raw anchor / positive / negative
- bottom row: augmented anchor / positive / negative
- footer: positive and negative mining details, quality band, score, and full paths

In [ ]:
examples = display_triplet_examples(
    context,
    num_examples=8,
    seed=SEED,
)
len(examples)

## Pair Keypoints And Matches

This uses the current RDD weights plus frozen LightGlue to show, for sampled triplets:
- detected keypoints on the positive pair
- detected keypoints on the negative pair
- positive pair matches
- negative pair matches

Use `device='cuda'` if you want GPU-backed matching.


In [ ]:
match_examples = display_triplet_match_examples(
    context,
    num_examples=4,
    seed=SEED,
    device='auto',
    max_matches=40,
    max_keypoints=256,
)
len(match_examples)


## Balanced Batch Overviews

When `finetune.sh` is in balanced mode, this shows the sampled training batch grouped as `n_classes x n_samples_per_class`.

In [ ]:
if context.args.batch_mode == 'balanced':
    batches = display_balanced_batch_overviews(
        context,
        num_batches=2,
        seed=SEED,
        show_augmented=True,
    )
    len(batches)
else:
    print('Current finetune.sh is not using balanced mode.')

## Raw Balanced Batches

Use this if you want to compare the augmented batch with the original images.

In [ ]:
if context.args.batch_mode == 'balanced':
    _ = display_balanced_batch_overviews(
        context,
        num_batches=1,
        seed=SEED,
        show_augmented=False,
    )